In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
from utils.metrics import qini_score

from models.gbdt_uplift_model import TwoStageGradientBoostingUpliftClassifier

DATASET_PATH = os.path.join(os.path.dirname(os.path.abspath(__file__)), "data")

def run_experiment(dataset_alias, fold_id=0):
    print(f"\n{'='*50}")
    print(f"RUNNING EXPERIMENT: {dataset_alias} (Fold {fold_id})")
    print(f"{'='*50}")
    
    # 1. Load Dữ liệu
    folder = os.path.join(DATASET_PATH, f'{dataset_alias}_{fold_id}')
    train_path = os.path.join(folder, 'train.pkl')
    test_path = os.path.join(folder, 'test.pkl')
    
    if not os.path.exists(train_path):
        print(f"[Error] Data not found at {train_path}.")
        return

    train = joblib.load(train_path)
    test = joblib.load(test_path)
    
    # Lấy ra các thành phần X, y, t
    X_train, y_train, t_train = train['X'], train['y'], train['t']
    X_test, y_test, t_test = test['X'], test['y'], test['t']

    # Xác định có bao nhiêu loại Treatment (loại bỏ số 0 là Control)
    available_treatments = np.unique(t_train)
    treatment_ids = available_treatments[available_treatments != 0]
    
    print(f"Train Shape: {X_train.shape}")
    print(f"Treatments Found: {treatment_ids}")

    results = {}
    
    # 2. Vòng lặp train riêng cho từng cặp (Control vs Treatment_i)
    # Vì model hiện tại thiết kế cho Binary Treatment
    for t_id in treatment_ids:
        print(f"\n--- Processing Treatment {t_id} vs Control ---")
        
        # 2a. Lọc dữ liệu Train (Chỉ lấy dòng Control + Treatment hiện tại)
        mask_train = np.isin(t_train, [0, t_id])
        X_tr_sub = X_train[mask_train]
        y_tr_sub = y_train[mask_train]
        t_tr_sub = t_train[mask_train]
        
        # Map lại nhãn treatment về 0 và 1 (Control=0, Treat_i=1)
        t_tr_binary = (t_tr_sub == t_id).astype(np.int32)
        
        # 2b. Khởi tạo & Train Model
        print("  > Training Model...")
        model = TwoStageGradientBoostingUpliftClassifier(
            learning_rate=0.05,
            max_depth=6,
            n_estimators=300,        # Số lượng cây (bạn có thể tăng lên 1000 nếu có GPU mạnh)
            uplift_ensemble_weight=0.5, # Trọng số kết hợp (50% tin vào Outcome, 50% tin vào Uplift trực tiếp)
            verbose=100              # In log mỗi 100 vòng
        )
        
        # Fit model
        model.fit(X_tr_sub, y_tr_sub, t_tr_binary)
        
        # 2c. Dự đoán & Đánh giá trên tập Test
        # Cũng lọc tập test tương ứng
        mask_test = np.isin(t_test, [0, t_id])
        X_test_sub = X_test[mask_test]
        y_test_sub = y_test[mask_test]
        t_test_sub = t_test[mask_test]
        t_test_binary = (t_test_sub == t_id).astype(np.int32)
        
        # Predict trả về Uplift Score
        pred_uplift = model.predict(X_test_sub)
        
        # Tính Qini Score
        score = qini_score(y_test_sub, pred_uplift, t_test_binary)
        results[f'Treatment_{t_id}'] = score
        print(f"  > Done. Qini Score: {score:.4f}")

    return results


In [ ]:
# --- Chạy thử Dataset Hillstrom (Fold 0) ---
# Hillstrom là dataset Marketing Email thực tế (Men's Email vs Women's Email)
scores_hillstrom = run_experiment('hillstrom', fold_id=0)
print("\nFinal Results (Hillstrom):", scores_hillstrom)

# --- Chạy thử Dataset Synthetic (Fold 0) ---
# Dataset giả lập với 6 loại treatment khác nhau
# scores_synth = run_experiment('synth1', fold_id=0)
# print("\nFinal Results (Synthetic):", scores_synth)